# V2.3 — BGE-M3 Bi-Encoder (No Rerank)

**Model:** `BAAI/bge-m3` — Multilingual SOTA 2024  
**Metrics:** Recall@1, @5, @100 | MRR@10

> ⚠ bge-m3 ~570M params. Cần ≥6GB VRAM.

In [1]:
import torch, numpy as np, faiss, csv as csv_mod
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from pipeline_utils import (
    get_eval_qa, write_jsonl, build_corpus,
    is_hit, build_result_entry,
    compute_metrics, print_metrics, save_csv,
    EVAL_DIR, TMP_DIR
)

VERSION    = "v2_3"
MODEL_NAME = "BAAI/bge-m3"
RESULTS_PATH = EVAL_DIR / f"pipeline_results_{VERSION}.jsonl"
CSV_PATH     = EVAL_DIR / f"metrics_{VERSION}.csv"
FAISS_PATH   = TMP_DIR  / f"faiss_{VERSION}.index"
TOP_N        = 100
BATCH        = 16
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model={MODEL_NAME} | Device={DEVICE} | TOP_N={TOP_N}")

d:\SGU\CNTT\NCKH2025_2026\ChatBot\cross-encoder\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model=BAAI/bge-m3 | Device=cuda | TOP_N=100


## 1. Load

In [2]:
bi_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
print(f"dim={bi_model.get_sentence_embedding_dimension()} ✓")
corpus  = build_corpus()
eval_qa = get_eval_qa()
print(f"Eval QA: {len(eval_qa)} queries")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 432.68it/s, Materializing param=pooler.dense.weight]                               


dim=1024 ✓
Corpus: 1840 passages
Eval QA: 570 queries


## 2. FAISS (reuse nếu có, rebuild nếu TOP_N thay đổi)

In [3]:
TMP_DIR.mkdir(parents=True, exist_ok=True)
if FAISS_PATH.exists():
    index = faiss.read_index(str(FAISS_PATH))
    print(f"FAISS loaded: {index.ntotal} vectors ✓")
else:
    print("Encoding corpus...")
    corpus_embs = bi_model.encode(
        [d["passage"] for d in corpus], batch_size=BATCH,
        normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True
    ).astype("float32")
    index = faiss.IndexFlatIP(corpus_embs.shape[1])
    index.add(corpus_embs)
    faiss.write_index(index, str(FAISS_PATH))
    print(f"FAISS built: {index.ntotal} vectors → saved")

Encoding corpus...


Batches: 100%|██████████| 115/115 [00:44<00:00,  2.60it/s]


FAISS built: 1840 vectors → saved


## 3. Evaluation

In [4]:
per_query = []
q_embs = bi_model.encode(
    [item["query"] for item in eval_qa], batch_size=BATCH,
    normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True
).astype("float32")
_, I = index.search(q_embs, TOP_N)

for item, top_ids_arr in tqdm(zip(eval_qa, I), total=len(eval_qa)):
    top_ids = top_ids_arr.tolist()
    ec = item["expected_citations"]
    rank_bi = next((r for r, idx in enumerate(top_ids, 1) if is_hit(idx, ec, corpus)), -1)
    hits_at = {k: 1 if any(is_hit(i, ec, corpus) for i in top_ids[:k]) else 0 for k in [1,3,5]}
    per_query.append(build_result_entry(
        item=item, top_ids=top_ids, corpus=corpus,
        score_map=None, rank_bi=rank_bi, rank_ce=rank_bi, hits_at=hits_at
    ))

recall100 = sum(1 for r in per_query if r["rank_bi"] > 0) / len(per_query)
print(f"Recall@100 (ceiling): {recall100:.4f}")

100%|██████████| 570/570 [00:00<00:00, 27869.78it/s]

Recall@100 (ceiling): 0.9246


## 4. Results

In [5]:
metrics = compute_metrics(per_query)
metrics["Recall@100"] = round(recall100, 4)
print_metrics(metrics, version_label=f"{VERSION} (bge-m3)")
write_jsonl(RESULTS_PATH, per_query)
save_csv(metrics, CSV_PATH, extra_cols={"version": VERSION, "bi_encoder": MODEL_NAME, "ce": "none"})
print("Saved ✓")


── v2_3 (bge-m3) Results ──
  Metric            Value
  ------------------------
  Recall@1         0.4719
  Recall@3         0.6579
  Recall@5         0.7228
  MRR@10           0.5685
  Recall@100       0.9246
  Saved → d:\SGU\CNTT\NCKH2025_2026\ChatBot\cross-encoder\notebooks\outputs\eval\metrics_v2_3.csv ✓
Saved ✓


## 5. Bảng so sánh V2.x (Decision gate)

In [6]:
print(f"{'Version':<6} {'Model':<34} {'R@1':>7} {'R@5':>7} {'R@100':>7} {'MRR':>7}")
print("-"*70)
best, best_r100 = None, 0
for ver, lbl in [("v2_1","multilingual-e5"),("v2_2","Vietnam_legal_HF"),("v2_3","bge-m3")]:
    f = EVAL_DIR / f"metrics_{ver}.csv"
    if not f.exists(): print(f"  {ver}: not run"); continue
    m = {r["metric"]: float(r["value"]) for r in csv_mod.DictReader(open(f))}
    flag = " ← best" if m.get("Recall@100",0) > best_r100 else ""
    if m.get("Recall@100",0) > best_r100:
        best_r100 = m["Recall@100"]; best = (ver, lbl)
    print(f"  {ver:<6} {lbl:<34} {m.get('Recall@1',0):>7.4f} {m.get('Recall@5',0):>7.4f} {m.get('Recall@100',0):>7.4f} {m.get('MRR@10',0):>7.4f}{flag}")
print(f"\n✅ Decision: Best bi-encoder = {best[1]} ({best[0]}) → dùng cho V3, V4, V5")

Version Model                                  R@1     R@5   R@100     MRR
----------------------------------------------------------------------
  v2_1   multilingual-e5                     0.4614  0.7035  0.9070  0.5543 ← best
  v2_2   Vietnam_legal_HF                    0.3175  0.5351  0.8298  0.4020
  v2_3   bge-m3                              0.4719  0.7228  0.9246  0.5685 ← best

✅ Decision: Best bi-encoder = bge-m3 (v2_3) → dùng cho V3, V4, V5
